# Section 4 — Practical 4: Swarm Pattern (Ollama + Qwen3)

**लक्ष्य:** एक काल्पनिक वीडियो गेम का Level बनाने के लिए 4 विशेषज्ञ AI Agents को साझा context के साथ एक-दूसरे को काम सौंपने दें।

**यह किस तरह अलग है?** Graph Pattern में पहले से तय रास्ते होते हैं। इस डेमो Swarm में हर Agent अगले उपलब्ध Specialist का सुझाव देता है। छोटा Qwen Model गलत चुनाव करे तो Python सुरक्षित fallback चुनता है। इस Notebook में सभी 4 विशेषज्ञों का योगदान अनिवार्य है; इसलिए यह **constrained swarm simulation** है, Strands SDK का native `Swarm` API नहीं।

**FIXED VERSION:** JSON output के लिए schema, automatic retry और Cell 7 में partial progress से resume जोड़ा गया है। अगर पुराने Notebook में Cell 7 बीच में रुका, तो इसी नए Notebook में Cell 1–6 क्रम से चलाएँ और फिर Cell 7।

**शुल्क:** local Ollama API से कोई paid API call नहीं। पहले `ollama run qwen3:1.7b` करके Qwen3 डाउनलोड होना चाहिए।

एक-एक **Code Cell** पर क्लिक करके **Shift + Enter** दबाएँ। कुल **8 code cells** हैं।

## Cell 1 — Ollama और Qwen3 Local Connection

In [ ]:
import json
from urllib.request import urlopen, Request
from urllib.error import URLError, HTTPError
from pathlib import Path
from datetime import datetime

OLLAMA_BASE = 'http://127.0.0.1:11434'
MODEL = 'qwen3:1.7b'

try:
    with urlopen(f'{OLLAMA_BASE}/api/tags', timeout=12) as response:
        tags = json.load(response)
except (URLError, TimeoutError, OSError) as exc:
    raise RuntimeError('Ollama server उपलब्ध नहीं। Windows में Ollama खोलें और फिर Cell 1 चलाएँ।') from exc

models = [m.get('name', '') for m in tags.get('models', [])]
print('Installed models:', models)
assert MODEL in models, f'{MODEL} नहीं मिला। PowerShell में चलाएँ: ollama pull {MODEL}'
print('PASS: Local Ollama + Qwen3 connected; AWS not required')

## Cell 2 — Game Brief और Shared Context

In [ ]:
BRIEF = {
    'game': 'Moonlight Maze',
    'genre': 'family-friendly puzzle adventure',
    'players': 'ages 10+',
    'level': 'Level 1 — Lantern Garden',
    'goal': '3 lanterns collect करके exit gate खोलना',
    'constraints': 'No violence, offline single-player, 5-minute level, simple keyboard controls',
}

print('FICTIONAL GAME:', BRIEF['game'])
print('LEVEL:', BRIEF['level'])
print('GOAL:', BRIEF['goal'])
assert len(BRIEF) == 6
print('PASS: Game design brief and shared context ready')

## Cell 3 — Local Ollama JSON Chat Function

In [ ]:
# Cell 3 (FIXED) — Reliable local Qwen3 JSON chat with retries
import json
import time
from urllib.request import urlopen, Request
from urllib.error import URLError, HTTPError

RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "contribution": {"type": "string"},
        "next_agent": {"type": "string"},
        "reason": {"type": "string"},
    },
    "required": ["contribution", "next_agent", "reason"],
}

def ollama_chat_json(system_prompt: str, user_prompt: str) -> dict:
    errors = []
    for attempt in range(1, 4):
        payload = {
            "model": MODEL,
            "stream": False,
            "think": False,
            "format": RESPONSE_SCHEMA,
            "messages": [
                {"role": "system", "content": system_prompt + " Respond ONLY with a short valid JSON object. No markdown."},
                {"role": "user", "content": user_prompt + "\nKeep contribution to 2-3 short Hindi sentences; reason under 15 words."},
            ],
            "options": {"temperature": 0, "num_predict": 1000, "num_ctx": 4096},
        }
        req = Request(
            f"{OLLAMA_BASE}/api/chat",
            data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
            headers={"Content-Type": "application/json"},
            method="POST",
        )
        try:
            with urlopen(req, timeout=300) as response:
                result = json.load(response)
            content = (result.get("message") or {}).get("content", "").strip()
            parsed = json.loads(content)
            if not isinstance(parsed, dict):
                raise ValueError("AI did not return a JSON object")
            if not str(parsed.get("contribution", "")).strip():
                raise ValueError("AI JSON has no contribution")
            parsed.setdefault("next_agent", "")
            parsed.setdefault("reason", "")
            return parsed
        except (json.JSONDecodeError, ValueError) as exc:
            errors.append(f"Attempt {attempt}: {type(exc).__name__}")
            print(f"JSON response incomplete, retry {attempt}/3")
            if attempt < 3:
                time.sleep(1)
        except (HTTPError, URLError, TimeoutError, OSError) as exc:
            raise RuntimeError(f"Ollama connection problem: {type(exc).__name__}: {exc}") from exc
    raise RuntimeError("Qwen3 returned invalid JSON after three attempts. " + "; ".join(errors))

print("PASS: Reliable Ollama JSON client ready (up to 3 retries, larger response limit)")


## Cell 4 — चार Specialist Agents की Roles

In [ ]:
AGENTS = {
    'level_designer': 'Create the core puzzle layout, goal, checkpoints and level flow.',
    'visual_artist': 'Describe visual art, palette, readability and accessible visual cues.',
    'gameplay_engineer': 'Specify player interaction, controls, progression and simple game logic.',
    'qa_tester': 'Identify edge cases, accessibility problems and a small manual test plan.',
}

START_AGENT = 'level_designer'
assert START_AGENT in AGENTS and len(AGENTS) == 4
for name, role in AGENTS.items():
    print(f'{name}: {role}')
print('PASS: Four specialist AI roles configured')

## Cell 5 — Handoff Validator और Offline Tests

In [ ]:
def validate_handoff(raw: dict, allowed_next: list[str]) -> tuple[str, str, bool]:
    """Accept the agent's next choice; fall back to first remaining role if invalid."""
    suggested = str(raw.get('next_agent', '')).strip().lower().replace(' ', '_')
    reason = str(raw.get('reason', 'Not specified')).strip()[:300]
    if not allowed_next:
        return 'FINISH', reason, False
    if suggested in allowed_next:
        return suggested, reason, False
    return allowed_next[0], 'Fallback: invalid or missing agent handoff', True

assert validate_handoff({'next_agent': 'qa_tester'}, ['qa_tester', 'visual_artist'])[0] == 'qa_tester'
assert validate_handoff({'next_agent': 'not_a_role'}, ['visual_artist'])[0] == 'visual_artist'
assert validate_handoff({'next_agent': 'visual_artist'}, [])[0] == 'FINISH'
print('PASS: Agent handoff validation and fallback tests')

## Cell 6 — पहला Real Qwen3 Specialist Call और Handoff

In [ ]:
def run_specialist(agent_name: str, prior_history: list[dict]) -> dict:
    visited = [entry['agent'] for entry in prior_history]
    remaining = [name for name in AGENTS if name not in visited and name != agent_name]
    prior_text = '\n'.join(
        f"- {entry['agent']}: {entry['contribution'][:220]}" for entry in prior_history
    ) or '(No prior contributions)'
    system = (
        f'You are {agent_name}, a specialist in a collaborative game-development swarm. '
        f'Your job: {AGENTS[agent_name]} '
        'Respond in simple Hindi. Use only this exact JSON object with keys '
        '"contribution", "next_agent", "reason". '
        'Contribution must be 2 short practical Hindi sentences (maximum 100 words). '
        'Choose next_agent from the permitted list; choose FINISH when list is empty. '
        'Do not invent facts about real games.'
    )
    allowed = remaining if remaining else ['FINISH']
    user = (
        f'GAME BRIEF: {json.dumps(BRIEF, ensure_ascii=False)}\n'
        f'SHARED WORK SO FAR:\n{prior_text}\n'
        f'YOUR SPECIALTY: {AGENTS[agent_name]}\n'
        f'PERMITTED next_agent VALUES: {allowed}\n'
        'Give your contribution and choose who should take the handoff next.'
    )
    raw = ollama_chat_json(system, user)
    contribution = str(raw.get('contribution', '')).strip()
    if not contribution:
        raise ValueError(f'{agent_name} returned no contribution; rerun this cell')
    next_agent, reason, fallback = validate_handoff(raw, remaining)
    return {
        'agent': agent_name,
        'contribution': contribution,
        'suggested_next_agent': str(raw.get('next_agent', '')),
        'next_agent': next_agent,
        'reason': reason,
        'fallback_used': fallback,
    }

history = [run_specialist(START_AGENT, [])]
first = history[0]
print('=== FIRST SPECIALIST ===')
print('Agent:', first['agent'])
print('Contribution:', first['contribution'])
print('Next handoff:', first['next_agent'])
print('PASS: First local Qwen3 specialist responded and proposed handoff')

## Cell 7 — Dynamic Swarm Handoffs (3 और Real AI Calls)

In [ ]:
assert 1 <= len(history) <= len(AGENTS), 'पहले Cell 6 चलाएँ।'
print(f'Resuming with {len(history)} completed specialist(s).')
MAX_AGENT_CALLS = len(AGENTS)

while len(history) < MAX_AGENT_CALLS:
    visited = {entry['agent'] for entry in history}
    remaining = [name for name in AGENTS if name not in visited]
    requested = history[-1]['next_agent']
    next_name = requested if requested in remaining else remaining[0]
    print(f'\nHANDOFF {len(history)}: {history[-1]["agent"]} -> {next_name}')
    entry = run_specialist(next_name, history)
    history.append(entry)
    print('Contribution:', entry['contribution'][:700])
    if entry['fallback_used']:
        print('Note: invalid suggested handoff; Python fallback was applied')

assert len(history) == 4
assert {entry['agent'] for entry in history} == set(AGENTS)
print('\nPASS: All 4 real Qwen3 specialist responses collected through bounded handoffs')
print('HANDOFF PATH:', ' -> '.join(item['agent'] for item in history), '-> FINISH')
print('Total fallback decisions:', sum(item['fallback_used'] for item in history))

## Cell 8 — Final Hindi Report और JSON Handoff Log Save

In [ ]:
assert len(history) == 4, 'Cell 7 चलाएँ ताकि सभी चार Agents रिपोर्ट दें।'

lines = [
    '# Moonlight Maze — AI Swarm Level Design Report',
    '',
    '> प्रशिक्षण के लिए काल्पनिक Level Design. चार local Qwen3 specialist calls.',
    '',
    f"**Game:** {BRIEF['game']}",
    f"**Level:** {BRIEF['level']}",
    f"**Goal:** {BRIEF['goal']}",
    '',
    '## Handoff sequence',
    ' → '.join(item['agent'] for item in history) + ' → FINISH',
    '',
]
for item in history:
    lines.extend([
        '## ' + item['agent'].replace('_', ' ').title(),
        item['contribution'],
        '',
        f"*Next handoff:* {item['next_agent']}",
        f"*Handoff reason:* {item['reason']}",
        '',
    ])
lines.extend([
    '## Notes',
    '- This is a bounded educational swarm, not the native Strands Swarm SDK.',
    '- Handoff suggestions come from the AI; Python validates them and falls back if needed.',
    '- This example does not create or ship a real video game.',
])

md_path = Path('section4_swarm_level_design_report.md')
json_path = Path('section4_swarm_handoff_log.json')
md_path.write_text('\n'.join(lines), encoding='utf-8')
json_path.write_text(json.dumps({'brief': BRIEF, 'handoffs': history}, ensure_ascii=False, indent=2), encoding='utf-8')
assert md_path.exists() and json_path.exists()
print('PASS: Section 4 Practical 4 — Swarm Pattern completed')
print('Saved:', md_path.resolve())
print('Saved:', json_path.resolve())
print('Fallback decisions:', sum(x['fallback_used'] for x in history))